# Fake vs. Real News Detection System (Optimized & Enhanced)

This notebook implements an institutional-grade machine learning pipeline to detect fake news. It features:
1. **Robust Dataset Ingestion**: Relative paths with auto-fallback to high-quality sample data to prevent file-not-found errors.
2. **Advanced NLP Preprocessing**: NLTK-based WordNet Lemmatization.
3. **Clickbait & Stylometric Feature Extraction**: Extracting structural tells like exclamation counts, question counts, title capitalization ratio, and title-to-body length ratios.
4. **Dual-Path Feature Fusion**: Vectorizing the title and body text separately to preserve title key-phrase density, and combining them with the scaled stylometric features.
5. **Ensemble Modeling**: Stacking Logistic Regression and a Passive-Aggressive Classifier in a Voting Classifier.
6. **Inference Bug Fix**: Resolved label mapping mismatches between strings and integers.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# Download required NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

## 1. Robust Dataset Ingestion
We load datasets using relative paths (`data/Fake.csv` and `data/True.csv`). If they do not exist, we automatically generate mock data to ensure seamless setup-free notebook runs.

In [ ]:
# Define relative paths
data_dir = "data"
fake_path = os.path.join(data_dir, "Fake.csv")
true_path = os.path.join(data_dir, "True.csv")

# Create data directory if missing
os.makedirs(data_dir, exist_ok=True)

# Fallback: Generate mock datasets if missing to allow out-of-the-box execution
if not os.path.exists(fake_path) or not os.path.exists(true_path):
    print("Dataset not found locally. Generating a mock dataset for seamless demonstration...")
    
    mock_fake_titles = [
        "BREAKING: SHOCKING evidence of alien base found on Moon!",
        "You won't believe what this politician did yesterday! CLICK HERE!",
        "ALERT: Vaccine contains microchips, secret source reveals!",
        "Government secretly planning to ban all pets by next month.",
        "SHOCKING: Secret cure for aging hidden by major companies!"
    ] * 200
    mock_fake_text = [
        "Unbelievable rumors are spreading across social media today about secret bases. Sources claim that officials are hiding the truth.",
        "In a video that has gone viral, a politician appears to make a shocking gesture. Viewers are outraged by this behavior.",
        "A leaked document from an anonymous insider suggests a massive conspiracy. Experts have not verified these claims.",
        "An anonymous blog post has sparked panic about new pet regulations. Pet owners are preparing to protest.",
        "Insiders claim that anti-aging formulas have been kept secret for decades to protect profits. No official statement was made."
    ] * 200
    
    mock_true_titles = [
        "Congress passes new environmental protection bill after debate.",
        "Scientists deploy new deep-sea ocean temperature sensor array.",
        "Central bank adjusts interest rates to manage inflation trends.",
        "Local community library opens new digital learning wing.",
        "City council approves funding for public transit upgrades."
    ] * 200
    mock_true_text = [
        "The legislation was approved by a bipartisan vote after weeks of intense negotiations on emissions standards.",
        "Researchers at the oceanographic institute successfully installed sensors at a depth of 4,000 meters to log temperatures.",
        "In its quarterly meeting, the board decided to increase the benchmark rate by a quarter point to stabilize the economy.",
        "The newly renovated facility will offer free coding bootcamps and access to high-speed internet for local residents.",
        "The budget allocation will support the acquisition of forty new electric buses and the expansion of two subway routes."
    ] * 200
    
    fake_df = pd.DataFrame({"title": mock_fake_titles, "text": mock_fake_text})
    true_df = pd.DataFrame({"title": mock_true_titles, "text": mock_true_text})
    
    fake_df.to_csv(fake_path, index=False)
    true_df.to_csv(true_path, index=False)
    print("Mock datasets successfully created in 'data/' folder!")

# Load datasets
fake = pd.read_csv(fake_path)
true = pd.read_csv(true_path)

# Standardize target label as binary integer (0 for FAKE, 1 for REAL)
fake['label'] = 0
true['label'] = 1

# Combine and shuffle datasets
combined = pd.concat([fake, true], axis=0)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Clean columns
combined = combined[['title', 'text', 'label']].dropna()
combined.to_csv("news_dataset.csv", index=False)
print(f"Dataset loaded. Total shape: {combined.shape}")

## 2. Advanced NLP Preprocessing & Lemmatization
Instead of just stripping words, we apply NLTK `WordNetLemmatizer` to normalize words to their root forms, reducing vocabulary sparsity.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

print("Applying lemmatization to title and body text...")
combined['clean_title'] = combined['title'].apply(preprocess_text)
combined['clean_text'] = combined['text'].apply(preprocess_text)
print("Preprocessing complete!")

## 3. Stylometric & Clickbait Feature Extraction
We extract metadata features such as title/text length, exclamation marks, question marks, and capitalization ratio in the title.

In [ ]:
def extract_stylometric_features(df):
    features = pd.DataFrame()
    
    # 1. Counts of exclamation marks
    features['title_excl_count'] = df['title'].apply(lambda x: str(x).count('!'))
    features['text_excl_count'] = df['text'].apply(lambda x: str(x).count('!'))
    
    # 2. Counts of question marks in title
    features['title_quest_count'] = df['title'].apply(lambda x: str(x).count('?'))
    
    # 3. Capitalization Ratio in Title (telltale sign of clickbait)
    def caps_ratio(title):
        words = str(title).split()
        if not words:
            return 0.0
        caps_words = sum(1 for w in words if w.isupper() and len(w) > 1)
        return caps_words / len(words)
        
    features['title_caps_ratio'] = df['title'].apply(caps_ratio)
    
    # 4. Article/Title length
    features['title_len'] = df['title'].apply(lambda x: len(str(x)))
    features['text_len'] = df['text'].apply(lambda x: len(str(x)))
    
    return features

print("Extracting clickbait and stylometric metrics...")
numerical_features = extract_stylometric_features(combined)
print("Features extracted!")
print(numerical_features.head())

## 4. Dual-Path Feature Fusion & Scaling
Vectorize the title and body text separately and merge them with scaled stylometric features.

In [ ]:
# Define distinct vectorizers
title_vectorizer = TfidfVectorizer(max_features=1500, ngram_range=(1, 2))
text_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

print("Performing dual-path TF-IDF vectorization...")
X_title_tfidf = title_vectorizer.fit_transform(combined['clean_title'])
X_text_tfidf = text_vectorizer.fit_transform(combined['clean_text'])

# Scale the numerical stylometric features
scaler = StandardScaler()
X_numerical_scaled = scaler.fit_transform(numerical_features)
X_numerical_sparse = csr_matrix(X_numerical_scaled)

# Concatenate all sparse representation matrices
X_all = hstack([X_title_tfidf, X_text_tfidf, X_numerical_sparse])
y = combined['label'].values

# Perform stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Fused feature matrices ready. Training size: {X_train.shape}")

## 5. Ensemble Classifier Training & Evaluation
Combine Logistic Regression and a Passive-Aggressive Classifier in a robust consensus Voting Classifier.

In [ ]:
# Define base learners
lr = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
pac = PassiveAggressiveClassifier(max_iter=100, random_state=42)

# Setup Voting ensemble
ensemble_model = VotingClassifier(
    estimators=[('lr', lr), ('pac', pac)],
    voting='hard'
)

print("Fitting Ensemble Model...")
ensemble_model.fit(X_train, y_train)
print("Model training completed.")

# Predict and evaluate performance
y_pred = ensemble_model.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['FAKE', 'REAL']))

## 6. Real-Time Inference & Corrected Prediction
We write the final, bug-free end-to-end prediction function mapping variables correctly and extracting full stylometric features.

In [ ]:
def predict_news(title, text):
    # Preprocess text fields
    clean_title = preprocess_text(title)
    clean_text = preprocess_text(text)
    
    # Build input features
    input_df = pd.DataFrame({"title": [title], "text": [text]})
    input_num = extract_stylometric_features(input_df)
    
    # Vectorize and scale
    vec_title = title_vectorizer.transform([clean_title])
    vec_text = text_vectorizer.transform([clean_text])
    vec_num = scaler.transform(input_num)
    vec_num_sparse = csr_matrix(vec_num)
    
    # Fuse together
    vec_all = hstack([vec_title, vec_text, vec_num_sparse])
    
    # Make consensus prediction
    pred = ensemble_model.predict(vec_all)[0]
    
    return "REAL NEWS" if pred == 1 else "FALSE NEWS"

# Run verification tests
print("--- VERIFICATION TESTS ---")
print("Test 1 (Fake News Specimen):")
print(predict_news(
    "SHOCKING EXPOSE: Alien base uncovered on the moon! MUST WATCH!",
    "Insiders leaked video showing strange structures. Click the link now to see full evidence!"
))

print("\nTest 2 (Real News Specimen):")
print(predict_news(
    "Bipartisan Committee Passes Milestone Infrastructure Spending Package",
    "The new allocation grants forty billion dollars for repairs to public transit networks and rail system upgrades."
))